# Milestone 2 - RLE and exploratory analysis
Notebook for GitHub: `notebooks/milestone-2-questions.ipynb` (branch `milestone-2`).

Covers: RLE encoding on an 8x8 grid (Q1), RLE decoding of one training image (Q2), class prevalence (Q3), defect co-occurrence (Q4). The printed lines at the end of each section are in the official answer format.

## Setup

In [ ]:
import os, glob, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IMG_H, IMG_W = 128, 800
DEFECTS = ["defect_a", "defect_b", "defect_c", "defect_d"]

KNOWN_TRAIN = "/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv"

def find_train_csv():
    """Use the known Kaggle path; otherwise search /kaggle/input and the working folder."""
    if os.path.exists(KNOWN_TRAIN):
        return KNOWN_TRAIN
    for root in ["/kaggle/input", "."]:
        hits = glob.glob(os.path.join(root, "**", "train.csv"), recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError("train.csv not found - attach the competition data to this notebook")

TRAIN_CSV = find_train_csv()
print("Using:", TRAIN_CSV)

## RLE utilities
Conventions: 1-based start index, column-major (Fortran) traversal. `rle_encode` closes runs with zero padding and detects value changes; `rle_decode` is its inverse and treats NaN/blank as an empty mask.

In [ ]:
def rle_encode(mask):
    """Binary 2D mask -> RLE string. 1-based start index, column-major (Fortran) order."""
    flat = np.asarray(mask).flatten(order="F")         
    padded = np.concatenate([[0], flat, [0]])         
    k = np.where(padded[1:] != padded[:-1])[0] + 1    
    starts, ends = k[0::2], k[1::2]                   
    return " ".join(f"{s} {e - s}" for s, e in zip(starts, ends))

def rle_decode(rle, shape=(IMG_H, IMG_W)):
    """RLE string -> binary (H, W) mask. NaN / empty / blank string -> all zeros."""
    flat = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    if isinstance(rle, str) and rle.strip():
        tok = np.array(rle.split(), dtype=np.int64)
        starts = tok[0::2] - 1                        
        ends = starts + tok[1::2]
        for s, e in zip(starts, ends):
            flat[max(s, 0):min(e, flat.size)] = 1     
    return flat.reshape(shape, order="F")             

In [ ]:
rng = np.random.default_rng(0)
for _ in range(200):
    m = (rng.random((IMG_H, IMG_W)) > 0.9).astype(np.uint8)
    assert (rle_decode(rle_encode(m)) == m).all()
print("round-trip test passed")

## Q1 - RLE of the 8 x 8 grid
Cells are numbered 1..64 down the columns. The highlighted cells were read from the figure of the question. The mask is encoded with `rle_encode` and cross-checked with an independent rule (group consecutive cell numbers into runs).

In [ ]:
HIGHLIGHTED = [12, 13, 23, 24, 25, 26, 44, 45, 47]

def cells_to_mask(cells, n=8):
    flat = np.zeros(n * n, dtype=np.uint8)
    flat[np.array(cells) - 1] = 1
    return flat.reshape((n, n), order="F")

def runs_from_cells(cells):
    cells = sorted(cells)
    runs, start, prev = [], cells[0], cells[0]
    for c in cells[1:]:
        if c != prev + 1:
            runs.append((start, prev - start + 1))
            start = c
        prev = c
    runs.append((start, prev - start + 1))
    return " ".join(f"{s} {l}" for s, l in runs)

grid = cells_to_mask(HIGHLIGHTED)
fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.imshow(grid, cmap="Greys", vmin=0, vmax=1.5)
for n in range(1, 65):
    r, c = (n - 1) % 8, (n - 1) // 8
    ax.text(c, r, str(n), ha="center", va="center", color="red" if grid[r, c] else "gray", fontsize=7)
ax.set_xticks([]); ax.set_yticks([])
plt.show()

assert rle_encode(grid) == runs_from_cells(HIGHLIGHTED)
print("Q1 rle_string:", rle_encode(grid))

## Q2 - decode the four masks of image `0ZUU8LH57Y`
Each defect row is decoded to a `(128, 800)` binary mask; empty entries (NaN) become all-zero masks. The answer is the sum of all positive pixels over the four masks.

In [ ]:
train = pd.read_csv(TRAIN_CSV)
print("rows:", len(train), "| images:", train["image_id"].nunique())

IMAGE_ID = "0ZUU8LH57Y"
rows = train[train["image_id"] == IMAGE_ID].set_index("defect_type").reindex(DEFECTS)
masks = {d: rle_decode(rows.loc[d, "mask_rle"]) for d in DEFECTS}
per_class = {d: int(m.sum()) for d, m in masks.items()}
assert all(m.shape == (IMG_H, IMG_W) for m in masks.values())
print("pixels per class:", per_class)
print("Q2 total positive pixels:", sum(per_class.values()))

## Q3 - class prevalence
A class is present in an image when its `mask_rle` is neither NaN nor blank. The file has exactly four rows per image, so counting present rows per class equals counting images per class.

In [ ]:
train["present"] = train["mask_rle"].fillna("").str.strip().ne("")
assert len(train) == 4 * train["image_id"].nunique()

counts = train.groupby("defect_type")["present"].sum().astype(int).sort_values(ascending=False)
print(counts.to_string())
print(f"Q3 most_common ({counts.index[0]}, {counts.iloc[0]}), rarest ({counts.index[-1]}, {counts.iloc[-1]})")

## Q4 - co-occurrence matrix
`P` is the binary image x class presence matrix; `C = P.T @ P` counts images containing both classes, with image counts on the diagonal.

In [ ]:
P = (train.pivot_table(index="image_id", columns="defect_type", values="present", aggfunc="max")
          .astype(int)[DEFECTS])
C = pd.DataFrame(P.values.T @ P.values, index=DEFECTS, columns=DEFECTS)
print(C.to_string())

assert (np.diag(C.values) == counts.reindex(DEFECTS).values).all()
pairs = {(i, j): int(C.loc[i, j]) for i, j in itertools.combinations(DEFECTS, 2)}
best = max(pairs, key=pairs.get)
assert ((P[best[0]] == 1) & (P[best[1]] == 1)).sum() == pairs[best]   # brute-force check
print(f"Q4 defect_pair, count: {best[0]} & {best[1]}, {pairs[best]}")